# Milestone 2 — Conditional Generation & Tuning

**Dataset:** `coco-processed` (milestone 1 output: 6,000 cleaned COCO captions over 3,000 images) ·
**Text encoder:** CLIP ViT-L/14 · **Diffusion model:** Stable Diffusion v1.5 · **Hardware:** Kaggle T4 ×2

The brief for this milestone:

1. Integrate a text encoder (CLIP) with a diffusion model (Stable Diffusion)
2. Run baseline conditional generation
3. Tune classifier-free guidance + noise schedule
4. Record the first set of generated images

| # | Section | Brief item |
|---|---|---|
| 0 | Setup — GPUs, packages, paths, run settings | |
| 1 | Fixed prompt set — 12 COCO captions and their real photos | |
| 2 | Load the models | |
| 3 | Integrate CLIP with the diffusion model through `prompt_embeds`, and verify it | **1** |
| 4 | The experiment harness — fixed latents, scheduler factory, scoring | |
| 5 | Baseline conditional generation | **2** |
| 6 | Classifier-free guidance sweep | **3** |
| 7 | Noise schedule — samplers, step counts, schedule shape, timestep spacing, Karras sigmas, DDPM | **3** |
| 8 | Results table and the selected configuration | |
| 9 | First set of generated images — final set, baseline vs best, denoising trajectory | **4** |
| 10 | Export | |

**How to run on Kaggle.** Attach the `coco-processed` dataset, set *Accelerator → GPU T4 ×2* and
*Internet → On*, then **Save Version → Save & Run All (Commit)**. A commit runs in the background, so
the browser can be closed, and everything the notebook writes ends up in the version's *Output* tab,
including `m2_outputs.zip`. A full run takes roughly an hour. Set `QUICK_RUN = True` in section 0 for
a 2–3 minute smoke test of every code path first.

**Re-running after a full run.** Add the earlier run's output as an input (*Add Input → Your Work →
Notebooks*, pick this notebook). Section 0 finds its `generations/` folder and copies the images in,
generation skips every image that already exists, and the re-run only scores them and generates what
is new — about 10 minutes instead of an hour.

**What makes these comparisons valid.** Every configuration generates the same 12 prompts from the
same starting noise, so the only thing that differs between two images in the same grid cell is the
setting being tested. Each configuration is scored numerically, not by eye.

---
## 0. Setup

In [ ]:
# --- packages -----------------------------------------------------------------
# Kaggle's image ships torch, transformers, scipy and usually diffusers. Install or upgrade
# diffusers only when it is missing or too old for `callback_on_step_end` / `guidance_rescale`.
import importlib, importlib.util, subprocess, sys

def _version(pkg):
    try:
        return importlib.import_module(pkg).__version__
    except Exception:
        return None

_dv = _version("diffusers")
if _dv is None or tuple(int(x) for x in _dv.split(".")[:2]) < (0, 27):
    print(f"diffusers {_dv} -> installing a current release")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers>=0.30", "accelerate"], check=True)
for pkg in ("accelerate", "scipy"):          # scipy: LMSDiscreteScheduler integrates with it
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

In [ ]:
import gc, json, os, platform, random, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import diffusers, transformers
from diffusers import (
    StableDiffusionPipeline, PNDMScheduler, DDIMScheduler, LMSDiscreteScheduler,
    EulerDiscreteScheduler, EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler,
    UniPCMultistepScheduler, DDPMScheduler,
)
from transformers import CLIPTokenizer, CLIPTextModel, CLIPModel, CLIPImageProcessor

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()
diffusers.logging.set_verbosity_error()

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}")
print(f"diffusers     {diffusers.__version__}")
print(f"transformers  {transformers.__version__}")
print()
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"cuda:{i}  {p.name}  {p.total_memory / 1024**3:.1f} GB")
else:
    print("No CUDA GPU found -- this will be very slow outside QUICK_RUN.")

### Run settings

Everything that changes the size or cost of the run is in the next cell. Two GPUs are used for two
different jobs: **`cuda:0` generates** and **`cuda:1` scores** (the CLIP evaluator), so the scorer
never competes with the UNet for memory.

In [ ]:
# ---- run size ----------------------------------------------------------------
QUICK_RUN  = False     # True: 2 prompts, 1 seed, <=3 steps, 256px -- a smoke test of every code path
SEED       = 42        # one source of randomness for prompt choice and starting noise
N_PROMPTS  = 12        # fixed prompt set, drawn from the curated subset
SEEDS      = [0, 1]    # two starting noises per prompt -> 24 images per configuration
FINAL_SEEDS = [0, 1, 2, 3]   # the recorded final set: 12 prompts x 4 seeds
GEN_BATCH  = 6         # images per pipeline call (x2 inside the UNet because of CFG)
RESOLUTION = 512       # SD 1.5's native resolution
PROMPT_MIN_WORDS, PROMPT_MAX_WORDS = 8, 18   # scene descriptions, not single objects

# ---- models ------------------------------------------------------------------
SD_REPO_CANDIDATES = [            # tried in order; the original runwayml repo was taken down
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "sd-legacy/stable-diffusion-v1-5",
]
TEXT_ENCODER_REPO = "openai/clip-vit-large-patch14"   # the encoder SD 1.5 was trained against
SCORER_REPO       = "openai/clip-vit-base-patch32"    # a *different* CLIP grades the images (as in M1)
CLIP_MAX_TOKENS   = 77

if QUICK_RUN:
    N_PROMPTS, SEEDS, FINAL_SEEDS, GEN_BATCH, RESOLUTION = 2, [0], [0, 1], 2, 256

def steps_for(n):
    # QUICK_RUN caps every step count so the smoke test stays fast.
    return min(n, 3) if QUICK_RUN else n

# ---- devices -----------------------------------------------------------------
if torch.cuda.is_available():
    GEN_DEVICE   = torch.device("cuda:0")
    SCORE_DEVICE = torch.device("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
    GEN_DTYPE    = torch.float16          # fp16 halves UNet memory and step time on a T4
elif torch.backends.mps.is_available():
    GEN_DEVICE = SCORE_DEVICE = torch.device("mps"); GEN_DTYPE = torch.float32
else:
    GEN_DEVICE = SCORE_DEVICE = torch.device("cpu"); GEN_DTYPE = torch.float32

# ---- paths -------------------------------------------------------------------
def find_processed_dir():
    # The folder holding pairs.parquet: an env override, the attached Kaggle dataset, or the repo's data/.
    if os.environ.get("M2_DATA_DIR"):
        return Path(os.environ["M2_DATA_DIR"])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        hits = sorted(kaggle_input.rglob("pairs.parquet"))
        if hits:
            return hits[0].parent
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / "processed" / "pairs.parquet").is_file():
            return root / "data" / "processed"
    raise FileNotFoundError("pairs.parquet not found -- attach the coco-processed dataset to this notebook")

PROCESSED_DIR = find_processed_dir()
IMAGES_512_DIR = PROCESSED_DIR / "images_512"
OUT_DIR = Path(os.environ.get("M2_OUT_DIR") or
               ("/kaggle/working/m2_outputs" if Path("/kaggle/working").is_dir() else "outputs/milestone_2"))
if QUICK_RUN:
    OUT_DIR = OUT_DIR.parent / (OUT_DIR.name + "_quick")
GEN_DIR, PLOTS_DIR, TABLES_DIR = OUT_DIR / "generations", OUT_DIR / "plots", OUT_DIR / "tables"
for d in (GEN_DIR, PLOTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- resume from a previous run ------------------------------------------------
# Attach an earlier run's output (Add Input -> your notebook, or the m2_outputs.zip uploaded as a
# dataset) and its images are copied in first. Generation skips any image that already exists, so
# a re-run only scores them and generates what is new.
def find_previous_generations():
    roots = [Path(os.environ["M2_PREVIOUS_RUN"])] if os.environ.get("M2_PREVIOUS_RUN") else []
    if Path("/kaggle/input").is_dir():
        roots.append(Path("/kaggle/input"))
    for root in roots:
        for d in sorted(root.rglob("generations")):
            if d.is_dir() and any(d.glob("*/p00_s0.png")):
                return d
    return None

PREVIOUS_GENERATIONS = None if QUICK_RUN else find_previous_generations()
if PREVIOUS_GENERATIONS:
    n_before = sum(1 for _ in GEN_DIR.rglob("*.png"))
    shutil.copytree(PREVIOUS_GENERATIONS, GEN_DIR, dirs_exist_ok=True)
    print(f"resumed from   : {PREVIOUS_GENERATIONS}  "
          f"({sum(1 for _ in GEN_DIR.rglob('*.png')) - n_before} images copied in)")

print(f"QUICK_RUN      : {QUICK_RUN}")
print(f"dataset        : {PROCESSED_DIR}")
print(f"outputs        : {OUT_DIR}")
print(f"generate on    : {GEN_DEVICE} ({str(GEN_DTYPE).replace('torch.', '')})   score on: {SCORE_DEVICE}")
print(f"per config     : {N_PROMPTS} prompts x {len(SEEDS)} seeds = {N_PROMPTS * len(SEEDS)} images at {RESOLUTION}px")

In [ ]:
# ---- plotting helpers ----------------------------------------------------------
# A fixed categorical order, assigned by entity and never cycled; grey is kept for reference lines.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
REF_GREY, INK, INK_2 = "#8a8985", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#ecebe7",
    "grid.linewidth": 0.8, "lines.linewidth": 2, "lines.markersize": 6,
})

def save_fig(fig, name):
    path = PLOTS_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return path

def save_table(name, obj):
    path = TABLES_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(json.dumps(obj, indent=2, default=float))
    return path

---
## 1. Fixed prompt set

The seniors' milestone 2 used one invented prompt per experiment. Here every experiment uses the same
**12 real COCO captions** from the milestone 1 subset: 8–18 words long (a scene, not a single object),
one caption per image, drawn with `SEED`. Each caption keeps its **real photo**, which gives two things a
made-up prompt cannot — a reference image to compare against, and a ceiling for the text–image score
(how well the real photo itself matches its caption).

In [ ]:
pairs = pd.read_parquet(PROCESSED_DIR / "pairs.parquet")
print(f"manifest: {len(pairs):,} captions over {pairs['image_id'].nunique():,} images")

pool = (pairs[pairs["n_words"].between(PROMPT_MIN_WORDS, PROMPT_MAX_WORDS)]
        .drop_duplicates("image_id")              # one caption per image -> 12 different scenes
        .reset_index(drop=True))
prompts_df = (pool.sample(N_PROMPTS, random_state=SEED)
              .reset_index(drop=True)
              .assign(prompt_idx=lambda d: range(len(d)),
                      real_path=lambda d: d["file_name"].map(lambda f: str(IMAGES_512_DIR / f))))
assert all(Path(p).is_file() for p in prompts_df["real_path"]), "a real image is missing from images_512/"

PROMPTS = prompts_df["caption"].tolist()
save_table("prompts.csv", prompts_df[["prompt_idx", "pair_idx", "image_id", "caption", "n_words", "file_name"]])
pd.set_option("display.max_colwidth", 120)
prompts_df[["prompt_idx", "image_id", "n_words", "caption"]]

In [ ]:
def show_real_photos(df, ncols=6):
    nrows = int(np.ceil(len(df) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.3 * ncols, 2.7 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, df.iterrows()):
        ax.imshow(Image.open(r["real_path"]))
        words = r["caption"].split()
        ax.set_title(f"[{r['prompt_idx']}] " + " ".join(words[:6]) + ("\n" + " ".join(words[6:12]) if len(words) > 6 else ""),
                     fontsize=7, fontweight="normal", color=INK_2)
    fig.suptitle("The fixed prompt set: real COCO photos and their captions", fontsize=11, color=INK)
    fig.tight_layout()
    return fig

fig = show_real_photos(prompts_df)
print("saved ->", save_fig(fig, "prompts_real_photos")); plt.show()

---
## 2. Load the models

| Part | Checkpoint | Role |
|---|---|---|
| Text encoder | `openai/clip-vit-large-patch14` (text tower) | turns a caption into 77 × 768 token embeddings |
| UNet | SD 1.5 | predicts the noise in a latent, cross-attending to those embeddings |
| VAE | SD 1.5 | decodes the 64 × 64 × 4 latent into a 512 × 512 image |
| Scheduler | swapped per experiment | decides which noise levels are visited and how each step is taken |
| Scorer | `openai/clip-vit-base-patch32`, on the second GPU | grades the results |

The text encoder is loaded **on its own**, from OpenAI's checkpoint, not taken from inside the SD pipeline,
so section 3 really does wire one model's output into the other. The scorer is a *different* CLIP from the
one driving generation, so the generator is not graded by its own encoder.

The safety checker is switched off: it replaces any image it flags with a black square, and its false
positives on ordinary COCO scenes would show up as zero scores in the metrics.

In [ ]:
def load_sd():
    last_err = None
    for repo in SD_REPO_CANDIDATES:
        for variant in (("fp16",) if GEN_DTYPE == torch.float16 else ()) + (None,):
            try:
                pipe = StableDiffusionPipeline.from_pretrained(
                    repo, torch_dtype=GEN_DTYPE, variant=variant,
                    safety_checker=None, requires_safety_checker=False)
                return repo, pipe
            except Exception as e:          # noqa: BLE001 -- try the next repo / variant
                last_err = e
    raise RuntimeError(f"could not load Stable Diffusion 1.5: {last_err}")

t0 = time.time()
SD_REPO, pipe = load_sd()
pipe = pipe.to(GEN_DEVICE)
pipe.set_progress_bar_config(disable=True)
# Decode the batch one image at a time to avoid a VRAM spike (the API moved between diffusers versions).
if hasattr(pipe.vae, "enable_slicing"):
    pipe.vae.enable_slicing()
elif hasattr(pipe, "enable_vae_slicing"):
    pipe.enable_vae_slicing()
BASE_SCHED_CONFIG = dict(pipe.scheduler.config)   # SD 1.5's trained schedule; every swap starts from it

tokenizer = CLIPTokenizer.from_pretrained(TEXT_ENCODER_REPO)
text_encoder = CLIPTextModel.from_pretrained(TEXT_ENCODER_REPO, torch_dtype=GEN_DTYPE).to(GEN_DEVICE).eval()

scorer = CLIPModel.from_pretrained(SCORER_REPO).to(SCORE_DEVICE).eval()
scorer_tok = CLIPTokenizer.from_pretrained(SCORER_REPO)
scorer_proc = CLIPImageProcessor.from_pretrained(SCORER_REPO)
print(f"loaded in {time.time() - t0:.0f}s  (SD repo: {SD_REPO})\n")

def n_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

components = pd.DataFrame([
    ("text encoder (ours, CLIP ViT-L/14)", n_params(text_encoder), str(GEN_DEVICE)),
    ("UNet", n_params(pipe.unet), str(GEN_DEVICE)),
    ("VAE", n_params(pipe.vae), str(GEN_DEVICE)),
    ("scorer (CLIP ViT-B/32)", n_params(scorer), str(SCORE_DEVICE)),
], columns=["component", "params (M)", "device"]).round(1)
print(f"scheduler shipped with SD 1.5: {BASE_SCHED_CONFIG['_class_name']}  "
      f"beta_schedule={BASE_SCHED_CONFIG['beta_schedule']}  steps_offset={BASE_SCHED_CONFIG.get('steps_offset')}")
components

---
## 3. Integrate the text encoder with the diffusion model

Normally the pipeline hides this step: `pipe(prompt="a dog")` tokenises and encodes the text internally.
Here we do that half ourselves and pass the result in through **`prompt_embeds`**, so the pipeline never
sees a string:

```
caption --CLIPTokenizer--> 77 token ids --CLIP text encoder--> 77 x 768 embeddings --prompt_embeds--> UNet cross-attention
```

The UNet's cross-attention layers expect exactly this shape — `cross_attention_dim = 768` — and,
more importantly, exactly this *space*: the UNet was trained against CLIP ViT-L/14's outputs and has
never seen any other encoder's vectors.

Classifier-free guidance needs a second, **unconditional** embedding: the encoding of the empty
string `""`. At every step the UNet predicts the noise twice, with and without the caption, and the
guidance scale *w* pushes the prediction away from the unconditional one:

&nbsp;&nbsp;&nbsp;&nbsp;ε̂ = ε(x<sub>t</sub>, ∅) + *w* · (ε(x<sub>t</sub>, caption) − ε(x<sub>t</sub>, ∅))

That is why `negative_prompt_embeds` is the empty-string encoding.

**Verification.** If the integration is correct, our embeddings equal what the pipeline computes
internally, and an image generated from `prompt_embeds` equals the one generated from `prompt=` with the
same starting noise.

In [ ]:
@torch.no_grad()
def encode_captions(texts):
    # Our side of the integration: caption strings -> (N, 77, 768) conditioning tensors.
    # No attention mask, matching how SD 1.5 was trained (its text encoder config does not use one).
    ids = tokenizer(texts, padding="max_length", max_length=CLIP_MAX_TOKENS,
                    truncation=True, return_tensors="pt").input_ids.to(GEN_DEVICE)
    return text_encoder(ids)[0]              # last_hidden_state, after the final layer norm

PROMPT_EMB = encode_captions(PROMPTS)        # (N_PROMPTS, 77, 768) -- computed once, used for every image
NEG_EMB = encode_captions([""])              # (1, 77, 768) -- the unconditional branch of CFG

print(f"prompt_embeds          : {tuple(PROMPT_EMB.shape)}  {PROMPT_EMB.dtype}")
print(f"negative_prompt_embeds : {tuple(NEG_EMB.shape)}")
print(f"UNet cross_attention_dim = {pipe.unet.config.cross_attention_dim}")
assert PROMPT_EMB.shape[-1] == pipe.unet.config.cross_attention_dim

# Check 1: our encoder against the encoder bundled inside the pipeline.
with torch.no_grad():
    pipe_pos, pipe_neg = pipe.encode_prompt(PROMPTS, GEN_DEVICE, 1, True, negative_prompt=None)
emb_diff = (PROMPT_EMB.float() - pipe_pos.float()).abs().max().item()
neg_diff = (NEG_EMB.float() - pipe_neg[:1].float()).abs().max().item()
emb_cos = F.cosine_similarity(PROMPT_EMB.float().flatten(1), pipe_pos.float().flatten(1)).min().item()
print(f"\nours vs pipeline's own encoder: max |diff| {emb_diff:.2e} (prompt), {neg_diff:.2e} (empty string); "
      f"min cosine {emb_cos:.6f}")

The image-level check needs the latent noise and scheduler helpers, so it runs right after the harness
is built, at the start of section 5.

---
## 4. The experiment harness

Three rules keep every comparison fair:

1. **Pre-generated starting noise.** Each (prompt, seed) pair gets its own latent tensor, drawn once from
   a **CPU** generator and passed into the pipeline. Fixing a seed is not enough on its own: different
   samplers consume random numbers differently, and `torch.Generator("cuda")` produces a different
   stream on different hardware. A CPU generator gives the same tensor on every machine.
2. **Separate noise for stochastic samplers.** Euler-ancestral and DDPM inject fresh noise at every
   step. They get their own per-image CPU generators, so their extra noise is reproducible too.
3. **Images are never overwritten.** Each configuration writes into its own folder and skips images
   that already exist, so an interrupted run resumes where it stopped, and a configuration that appears
   in two experiments (the baseline appears in several) is generated once.

**Scoring** — every image gets three numbers, all from the separate ViT-B/32 scorer:

| Metric | What it measures |
|---|---|
| `clip_text` | cosine between the image and its caption — prompt adherence. Real matching pairs sit around 0.25–0.35 |
| `clip_real` | cosine between the image and the real COCO photo for that caption — does it depict the same scene |
| `saturation` | mean HSV saturation, 0–1 — a direct measure of the over-saturation high CFG is known for |
| `sharpness` | variance of the Laplacian — edge and texture detail. CLIP scores barely react to blur, so too few steps can look fine to CLIP and soft to a person; this catches it |

Seconds per image is recorded as well. Each configuration is also compared with the baseline **pair by
pair** — same prompt, same starting noise — which removes the prompt-to-prompt variation and makes the
differences far more reliable than comparing two averages.

In [ ]:
# ---- starting noise -------------------------------------------------------------
LATENT_CH = pipe.unet.config.in_channels
LATENT_HW = RESOLUTION // pipe.vae_scale_factor          # 512 / 8 = 64

def start_latent(prompt_idx, seed):
    g = torch.Generator("cpu").manual_seed(SEED * 100_000 + seed * 1_000 + prompt_idx)
    return torch.randn((1, LATENT_CH, LATENT_HW, LATENT_HW), generator=g, dtype=torch.float32)

def step_noise_generator(prompt_idx, seed):
    # a separate stream for samplers that add noise during the loop (Euler-a, DDPM)
    return torch.Generator("cpu").manual_seed(SEED * 100_000 + seed * 1_000 + prompt_idx + 50_000)

# ---- schedulers -------------------------------------------------------------------
# Every scheduler is built from SD 1.5's own config, so it inherits the trained beta schedule.
# Experiments then override one setting at a time.
SAMPLERS = {
    "pndm":     (PNDMScheduler, {}),                       # SD 1.5's default
    "ddim":     (DDIMScheduler, {}),                       # deterministic (eta = 0)
    "lms":      (LMSDiscreteScheduler, {}),                # linear multistep
    "euler":    (EulerDiscreteScheduler, {}),
    "euler_a":  (EulerAncestralDiscreteScheduler, {}),     # ancestral: adds noise every step
    "dpmpp_2m": (DPMSolverMultistepScheduler, {"algorithm_type": "dpmsolver++", "solver_order": 2}),
    "unipc":    (UniPCMultistepScheduler, {}),
    "ddpm":     (DDPMScheduler, {}),                       # the original ancestral sampler
}
SAMPLER_LABELS = {"pndm": "PNDM", "ddim": "DDIM", "lms": "LMS", "euler": "Euler", "euler_a": "Euler-a",
                  "dpmpp_2m": "DPM++ 2M", "unipc": "UniPC", "ddpm": "DDPM"}

def make_scheduler(sampler, **overrides):
    cls, defaults = SAMPLERS[sampler]
    return cls.from_config(BASE_SCHED_CONFIG, **{**defaults, **overrides})

# ---- configurations ------------------------------------------------------------------
def C(sampler, steps, cfg=7.5, rescale=0.0, **sched):
    # One generation setting. Its name doubles as its output folder.
    steps = steps_for(steps)
    name = f"{sampler}_s{steps}_g{cfg:g}"
    if rescale:
        name += f"_r{rescale:g}"
    for k, v in sorted(sched.items()):
        short = {"timestep_spacing": "spacing", "beta_schedule": "beta", "use_karras_sigmas": "karras"}.get(k, k)
        name += f"_{short}-{v}"
    return {"name": name, "sampler": sampler, "steps": steps, "cfg": float(cfg),
            "rescale": float(rescale), "sched": sched}

# ---- generation ----------------------------------------------------------------------
def image_path(cfg, prompt_idx, seed):
    return GEN_DIR / cfg["name"] / f"p{prompt_idx:02d}_s{seed}.png"

def generate(cfg, prompt_ids=None, seeds=None):
    # Generate every missing (prompt, seed) image for one configuration. Returns how many were new.
    prompt_ids = list(range(N_PROMPTS)) if prompt_ids is None else list(prompt_ids)
    seeds = SEEDS if seeds is None else seeds
    out_dir = GEN_DIR / cfg["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    timing_file = out_dir / "timing.json"
    timing = json.loads(timing_file.read_text()) if timing_file.is_file() else {}

    jobs = [(p, s) for s in seeds for p in prompt_ids if not image_path(cfg, p, s).is_file()]
    if not jobs:
        return 0
    pipe.scheduler = make_scheduler(cfg["sampler"], **cfg["sched"])
    for i in range(0, len(jobs), GEN_BATCH):
        batch = jobs[i:i + GEN_BATCH]
        idx = [p for p, _ in batch]
        latents = torch.cat([start_latent(p, s) for p, s in batch]).to(GEN_DEVICE, GEN_DTYPE)
        if GEN_DEVICE.type == "cuda":
            torch.cuda.synchronize(GEN_DEVICE)
        t0 = time.perf_counter()
        images = pipe(
            prompt_embeds=PROMPT_EMB[idx],
            negative_prompt_embeds=NEG_EMB.expand(len(batch), -1, -1),
            latents=latents,
            generator=[step_noise_generator(p, s) for p, s in batch],
            num_inference_steps=cfg["steps"],
            guidance_scale=cfg["cfg"],
            guidance_rescale=cfg["rescale"],
            height=RESOLUTION, width=RESOLUTION,
        ).images
        if GEN_DEVICE.type == "cuda":
            torch.cuda.synchronize(GEN_DEVICE)
        per_image = (time.perf_counter() - t0) / len(batch)
        for (p, s), img in zip(batch, images):
            img.save(image_path(cfg, p, s))
            timing[image_path(cfg, p, s).name] = per_image
        timing_file.write_text(json.dumps(timing))
    return len(jobs)

# ---- scoring ---------------------------------------------------------------------------
def _pooled(out):
    # transformers >= 5 returns an output object from get_*_features; older versions a tensor
    return out.pooler_output if hasattr(out, "pooler_output") else out

@torch.no_grad()
def image_features(images, batch_size=32):
    feats = []
    for i in range(0, len(images), batch_size):
        batch = [im if isinstance(im, Image.Image) else Image.open(im) for im in images[i:i + batch_size]]
        px = scorer_proc(images=[im.convert("RGB") for im in batch], return_tensors="pt").to(SCORE_DEVICE)
        feats.append(F.normalize(_pooled(scorer.get_image_features(**px)).float(), dim=-1).cpu())
    return torch.cat(feats)

@torch.no_grad()
def text_features(texts):
    toks = scorer_tok(texts, padding="max_length", max_length=CLIP_MAX_TOKENS,
                      truncation=True, return_tensors="pt").to(SCORE_DEVICE)
    return F.normalize(_pooled(scorer.get_text_features(**toks)).float(), dim=-1).cpu()

def saturation(img):
    return float(np.asarray(img.convert("HSV"), dtype=np.float32)[..., 1].mean() / 255.0)

def sharpness(img):
    # Variance of the Laplacian on the greyscale image: high for crisp edges and fine texture, low for
    # blur. CLIP scores barely notice blur, which is why this is measured separately.
    g = np.asarray(img.convert("L"), dtype=np.float32)
    lap = 4 * g[1:-1, 1:-1] - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    return float(lap.var())

TEXT_FEAT = text_features(PROMPTS)
REAL_FEAT = image_features(prompts_df["real_path"].tolist())

def score(cfg, group, prompt_ids=None, seeds=None):
    # One row per image of one configuration.
    prompt_ids = list(range(N_PROMPTS)) if prompt_ids is None else list(prompt_ids)
    seeds = SEEDS if seeds is None else seeds
    keys = [(p, s) for s in seeds for p in prompt_ids]
    paths = [image_path(cfg, p, s) for p, s in keys]
    images = [Image.open(x).convert("RGB") for x in paths]
    feats = image_features(images)
    timing_file = GEN_DIR / cfg["name"] / "timing.json"
    timing = json.loads(timing_file.read_text()) if timing_file.is_file() else {}
    rows = []
    for (p, s), path, img, f in zip(keys, paths, images, feats):
        rows.append({
            "config": cfg["name"], "group": group, "sampler": cfg["sampler"], "steps": cfg["steps"],
            "cfg": cfg["cfg"], "rescale": cfg["rescale"],
            "sched": ",".join(f"{k}={v}" for k, v in sorted(cfg["sched"].items())) or "-",
            "prompt_idx": p, "seed": s,
            "clip_text": float(f @ TEXT_FEAT[p]), "clip_real": float(f @ REAL_FEAT[p]),
            "saturation": saturation(img), "sharpness": sharpness(img),
            "sec_per_img": timing.get(path.name, np.nan),
        })
    return rows

# ---- running experiments -----------------------------------------------------------------
RESULTS = []           # every scored image from every experiment

def run(configs, group, prompt_ids=None, seeds=None):
    # Generate (skipping what already exists) and score a list of configurations.
    configs = list({c["name"]: c for c in configs}.values())   # QUICK_RUN can collapse two configs into one
    rows = []
    for c in tqdm(configs, desc=group):
        generate(c, prompt_ids, seeds)
        rows += score(c, group, prompt_ids, seeds)
    RESULTS.extend(rows)
    pd.DataFrame(RESULTS).to_csv(TABLES_DIR / "results_per_image.csv", index=False)
    return pd.DataFrame(rows)

# The real photos, scored the same way: a reference row for every plot.
REAL_REF = {
    "clip_text": float((REAL_FEAT * TEXT_FEAT).sum(-1).mean()),
    "saturation": float(np.mean([saturation(Image.open(p).convert("RGB")) for p in prompts_df["real_path"]])),
    "sharpness": float(np.mean([sharpness(Image.open(p).convert("RGB")) for p in prompts_df["real_path"]])),
}
print(f"real COCO photos vs their own captions: clip_text {REAL_REF['clip_text']:.4f}, "
      f"saturation {REAL_REF['saturation']:.3f}, sharpness {REAL_REF['sharpness']:.0f}")

In [ ]:
# ---- summaries ------------------------------------------------------------------------
METRICS = ["clip_text", "clip_real", "saturation", "sharpness"]

def summarise(df, baseline_name=None):
    # One row per configuration: mean and standard error, plus the paired difference from the baseline.
    def sem(x):
        return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
    keys = ["config", "group", "sampler", "steps", "cfg", "rescale", "sched"]
    g = df.groupby(keys, sort=False)
    out = g.agg(n=("clip_text", "size"),
                **{f"{m}": (m, "mean") for m in METRICS},
                **{f"{m}_sem": (m, sem) for m in METRICS},
                sec_per_img=("sec_per_img", "mean")).reset_index()
    if baseline_name is not None:
        base = (pd.DataFrame(RESULTS).query("config == @baseline_name")
                .drop_duplicates(["prompt_idx", "seed"]).set_index(["prompt_idx", "seed"])["clip_text"])
        deltas = []
        for name, sub in df.groupby("config", sort=False):
            d = sub.set_index(["prompt_idx", "seed"])["clip_text"] - base
            d = d.dropna()
            half = 1.96 * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
            deltas.append({"config": name, "delta_clip_text": d.mean(), "delta_ci95": half})
        out = out.merge(pd.DataFrame(deltas), on="config", how="left")
    return out

def fmt(summary, cols=("config", "n", "clip_text", "clip_text_sem", "delta_clip_text", "delta_ci95",
                       "clip_real", "saturation", "sharpness", "sec_per_img")):
    cols = [c for c in cols if c in summary.columns]
    return summary[cols].round(4)

# ---- image grids -------------------------------------------------------------------------
GRID_PROMPTS = list(range(min(4, N_PROMPTS)))

def image_grid(configs, labels, title, prompt_ids=None, seed=None, with_real=True, cell=2.1):
    # Rows are prompts, columns are configurations; the first column is the real photo.
    prompt_ids = GRID_PROMPTS if prompt_ids is None else prompt_ids
    seed = SEEDS[0] if seed is None else seed
    ncols = len(configs) + int(with_real)
    fig, axes = plt.subplots(len(prompt_ids), ncols, figsize=(cell * ncols, cell * len(prompt_ids) + 0.6),
                             squeeze=False)
    for r, p in enumerate(prompt_ids):
        row = ([Image.open(prompts_df.loc[p, "real_path"])] if with_real else []) + \
              [Image.open(image_path(c, p, seed)) for c in configs]
        heads = (["real photo"] if with_real else []) + list(labels)
        for c, (img, head) in enumerate(zip(row, heads)):
            ax = axes[r, c]
            ax.imshow(img); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(False)
            if r == 0:
                ax.set_title(head, fontsize=8, fontweight="bold", color=INK)
        axes[r, 0].set_ylabel(f"[{p}] " + " ".join(PROMPTS[p].split()[:4]) + "...", fontsize=7, color=INK_2)
    fig.suptitle(title, fontsize=11, color=INK)
    fig.tight_layout()
    return fig

---
## 5. Baseline conditional generation

The control every other configuration is measured against, and the same settings milestone 1 used:
**PNDM, 30 steps, CFG 7.5.** PNDM is the scheduler SD 1.5 ships with; 7.5 is its default guidance.

First, the image-level integration check promised in section 3: the same two prompts and the same
starting noise, once with `prompt=` (the pipeline encodes the text) and once with our `prompt_embeds`.

In [ ]:
BASELINE = C("pndm", 30, 7.5)

# Integration check 2: prompt= versus prompt_embeds=, same starting noise, same scheduler.
check_ids = list(range(min(2, N_PROMPTS)))
pipe.scheduler = make_scheduler("pndm")
lat = torch.cat([start_latent(p, SEEDS[0]) for p in check_ids]).to(GEN_DEVICE, GEN_DTYPE)
common = dict(latents=lat, num_inference_steps=BASELINE["steps"], guidance_scale=7.5,
              height=RESOLUTION, width=RESOLUTION, output_type="np")
via_text = pipe(prompt=[PROMPTS[p] for p in check_ids], **common).images
# Encode with exactly the batch shapes the pipeline uses internally (2 captions, 2 empty strings).
# In fp16 on a GPU the matrix-multiply kernel, and so the last bit of rounding, depends on batch
# size; encoding the empty string once and copying it gave a 1-ulp difference on the first run.
check_pos, check_neg = encode_captions([PROMPTS[p] for p in check_ids]), encode_captions([""] * len(check_ids))
via_embeds = pipe(prompt_embeds=check_pos, negative_prompt_embeds=check_neg, **common).images
pix = np.abs(via_text - via_embeds) * 255
integration = {
    "embedding_max_abs_diff": emb_diff, "negative_embedding_max_abs_diff": neg_diff,
    "embedding_min_cosine": emb_cos,
    "image_max_abs_diff_0_255": float(pix.max()), "image_mean_abs_diff_0_255": float(pix.mean()),
    "pixel_identical": bool(pix.max() == 0),
}
save_table("integration_check.json", integration)
for k, v in integration.items():
    print(f"  {k:<34}{v}")
# Verdict. The embeddings are the real test: if they agree to fp16 precision, the conditioning is the
# same. Pixel differences on top of that are rounding amplified over 30 sampler steps -- they show up
# as small local changes in detail, with the same composition (see the figure).
same_embeddings = emb_diff < 0.05 and emb_cos > 0.9999
if integration["pixel_identical"]:
    verdict = "PASS: prompt_embeds reproduces prompt= pixel for pixel."
elif same_embeddings and pix.mean() < 5.0:
    verdict = ("PASS: same embeddings and the same picture; the remaining pixel differences are fp16 "
               "rounding amplified by the sampler.")
else:
    verdict = "CHECK: the embeddings or the images differ by more than rounding -- inspect the encoders."
integration["verdict"] = verdict
save_table("integration_check.json", integration)
print("\n" + verdict)

fig, axes = plt.subplots(len(check_ids), 2, figsize=(5, 2.6 * len(check_ids)), squeeze=False)
for r, p in enumerate(check_ids):
    for c, (img, t) in enumerate([(via_text[r], 'prompt="..."'), (via_embeds[r], "prompt_embeds (ours)")]):
        axes[r, c].imshow(img); axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(t, fontsize=9)
fig.suptitle("Integration check: pipeline's text path vs our encoder", fontsize=10)
fig.tight_layout(); print("saved ->", save_fig(fig, "integration_check")); plt.show()

In [ ]:
baseline_df = run([BASELINE], "baseline")
baseline_summary = summarise(baseline_df)
save_table("baseline_summary.csv", baseline_summary)
print(f"Baseline ({BASELINE['name']}), {len(baseline_df)} images")
print(f"  clip_text  {baseline_summary.clip_text[0]:.4f} +/- {baseline_summary.clip_text_sem[0]:.4f}   "
      f"(real photos: {REAL_REF['clip_text']:.4f})")
print(f"  clip_real  {baseline_summary.clip_real[0]:.4f}")
print(f"  saturation {baseline_summary.saturation[0]:.3f}   (real photos: {REAL_REF['saturation']:.3f})")
print(f"  time       {baseline_summary.sec_per_img[0]:.2f} s/image")

# Baseline grid: every prompt, both seeds.
fig, axes = plt.subplots(len(SEEDS) + 1, N_PROMPTS, figsize=(1.7 * N_PROMPTS, 1.8 * (len(SEEDS) + 1) + 0.5), squeeze=False)
for p in range(N_PROMPTS):
    rows = [Image.open(prompts_df.loc[p, "real_path"])] + [Image.open(image_path(BASELINE, p, s)) for s in SEEDS]
    for r, img in enumerate(rows):
        ax = axes[r, p]; ax.imshow(img); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for sp in ax.spines.values():
            sp.set_visible(False)
    axes[0, p].set_title(f"[{p}]", fontsize=8)
for r, lab in enumerate(["real"] + [f"seed {s}" for s in SEEDS]):
    axes[r, 0].set_ylabel(lab, fontsize=8, color=INK_2)
fig.suptitle("Baseline (PNDM, 30 steps, CFG 7.5): real photo on top, generated below", fontsize=11, color=INK)
fig.tight_layout(); print("saved ->", save_fig(fig, "baseline_grid")); plt.show()

---
## 6. Classifier-free guidance sweep

The guidance scale *w* is how hard each step is pushed toward the caption and away from the
unconditional prediction. *w* = 1 is plain conditional sampling (no guidance — the pipeline skips the
unconditional pass entirely); larger values trade diversity and naturalness for prompt adherence, and
past a point colours burn out and contrast blows up.

**Guidance rescale** (`guidance_rescale`, Lin et al. 2023) is the standard fix for that burn-out: it
rescales the guided prediction back toward the standard deviation of the conditional one. It is tested
at the high end of the range, where it matters.

Everything else is held at the baseline: PNDM, 30 steps, same prompts, same starting noise.

**Note on cost:** the seniors reported CFG 15 as "7× slower". It cannot be: the guidance scale is a
number in one multiply-add, and the UNet does the same two passes per step at any *w* > 1. The timing
column below measures it.

In [ ]:
CFG_VALUES = [1.0, 3.0, 5.0, 7.5, 10.0, 12.0, 15.0]
RESCALE_AT = [10.0, 15.0]
cfg_configs = [C("pndm", 30, g) for g in CFG_VALUES] + [C("pndm", 30, g, rescale=0.7) for g in RESCALE_AT]

cfg_df = run(cfg_configs, "cfg")
cfg_summary = summarise(cfg_df, BASELINE["name"])
save_table("cfg_summary.csv", cfg_summary)
fmt(cfg_summary)

In [ ]:
plain = cfg_summary[cfg_summary.rescale == 0].sort_values("cfg")
resc = cfg_summary[cfg_summary.rescale > 0].sort_values("cfg")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
panels = [("clip_text", "Prompt adherence (CLIP text-image cosine)"),
          ("clip_real", "Similarity to the real COCO photo"),
          ("saturation", "Colour saturation (HSV, 0-1)")]
for ax, (m, title) in zip(axes, panels):
    ax.errorbar(plain.cfg, plain[m], yerr=plain[f"{m}_sem"], color=PALETTE[0], marker="o", capsize=3,
                label="CFG only")
    ax.errorbar(resc.cfg, resc[m], yerr=resc[f"{m}_sem"], color=PALETTE[1], marker="s", capsize=3,
                linestyle="none", label="CFG + guidance_rescale 0.7")
    if m in REAL_REF:
        ax.axhline(REAL_REF[m], color=REF_GREY, linestyle="--", linewidth=1.2)
        ax.annotate("real photos", (ax.get_xlim()[1], REAL_REF[m]), fontsize=7, color=INK_2,
                    ha="right", va="bottom")
    ax.axvline(7.5, color=REF_GREY, linewidth=0.8, linestyle=":")
    ax.set_title(title); ax.set_xlabel("guidance scale w"); ax.set_xticks(CFG_VALUES)
axes[0].legend(loc="lower right")
fig.suptitle("Classifier-free guidance sweep -- PNDM, 30 steps; mean +/- s.e. over "
             f"{N_PROMPTS * len(SEEDS)} images (dotted line: baseline w = 7.5)", fontsize=10, color=INK)
fig.tight_layout(); print("saved ->", save_fig(fig, "cfg_sweep_metrics")); plt.show()

fig = image_grid([C("pndm", 30, g) for g in CFG_VALUES] + [C("pndm", 30, 15.0, rescale=0.7)],
                 [f"w = {g:g}" for g in CFG_VALUES] + ["w = 15\n+ rescale 0.7"],
                 "Guidance scale: same prompt, same starting noise, only w changes")
print("saved ->", save_fig(fig, "cfg_sweep_grid")); plt.show()

---
## 7. Noise schedule

"Noise schedule" covers two different things, and both are tuned here:

- **The schedule itself** — the curve of noise levels ᾱ<sub>t</sub> (how much of the original signal
  survives at timestep *t*) over the 1,000 training timesteps, set by `beta_schedule`.
- **How inference walks that schedule** — the **sampler** (the numerical method for each step), the
  **number of steps**, which timesteps are visited (`timestep_spacing`), and how the noise levels are
  spaced (Karras sigmas).

Every experiment here keeps CFG at the baseline 7.5, so the CFG and schedule effects stay separate.

### 7a. What the knobs change

Before generating anything: what the schedules and spacings actually look like.

In [ ]:
def alphas_cumprod(beta_schedule):
    s = DDPMScheduler.from_config(BASE_SCHED_CONFIG, beta_schedule=beta_schedule)
    return s.alphas_cumprod.numpy()

BETA_SCHEDULES = ["scaled_linear", "linear", "squaredcos_cap_v2"]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for i, b in enumerate(BETA_SCHEDULES):
    ac = alphas_cumprod(b)
    label = b + (" (SD 1.5 trained)" if b == BASE_SCHED_CONFIG["beta_schedule"] else "")
    axes[0].plot(ac, color=PALETTE[i], label=label)
    axes[1].plot(np.log(ac / (1 - ac)), color=PALETTE[i], label=label)
axes[0].set_title("Signal kept at timestep t (alpha-bar)"); axes[0].set_xlabel("timestep t")
axes[1].set_title("log signal-to-noise ratio"); axes[1].set_xlabel("timestep t")
axes[0].legend(loc="upper right")

N_SHOW = 10
for i, spacing in enumerate(["leading", "linspace", "trailing"]):
    s = DDIMScheduler.from_config(BASE_SCHED_CONFIG, timestep_spacing=spacing)
    s.set_timesteps(N_SHOW)
    t = s.timesteps.numpy()
    axes[2].scatter(t, np.full_like(t, i), color=PALETTE[i], s=36, zorder=3)
    axes[2].annotate(f"first t = {t[0]}", (t[0], i + 0.25), fontsize=7, color=INK_2, ha="right")
axes[2].set_yticks([0, 1, 2], ["leading (SD default)", "linspace", "trailing"])
axes[2].set_xlim(-20, 1020); axes[2].set_ylim(-0.6, 2.7)
axes[2].set_title(f"Timesteps visited with {N_SHOW} steps"); axes[2].set_xlabel("timestep t")
fig.tight_layout(); print("saved ->", save_fig(fig, "schedule_curves")); plt.show()

ac = alphas_cumprod("scaled_linear")
print(f"SD 1.5's schedule leaves alpha-bar = {ac[-1]:.5f} at t = 999: the last training step still holds "
      f"{np.sqrt(ac[-1]) * 100:.1f}% of the signal amplitude.")

**Reading the plots.** The three beta schedules destroy the signal at very different rates. A pretrained
UNet has learned what an image looks like at each noise level *of its own schedule*, so swapping the
schedule at inference time tells the UNet the wrong noise level at every step — section 7d measures what
that costs. The timestep panel shows why spacing matters at low step counts: SD's default `leading`
spacing never starts at the noisiest timestep (it starts at 901 with 10 steps), so the first step denoises
from a noise level the starting latent does not actually have.

**Not tested — `rescale_betas_zero_snr`.** It forces ᾱ to exactly zero at the last timestep. That only
works for v-prediction models trained that way; SD 1.5 predicts noise (ε), and at zero signal the
ε-to-image conversion divides by zero. Applying it to SD 1.5 produces NaNs or garbage, so it is left out.

### 7b. Sampler comparison

Seven samplers, all at **30 steps** and CFG 7.5 from the same starting noise. PNDM at 30 steps is the
baseline itself.

In [ ]:
SAMPLER_SWEEP = ["pndm", "ddim", "lms", "euler", "euler_a", "dpmpp_2m", "unipc"]
sampler_configs = [C(s, 30) for s in SAMPLER_SWEEP]
sampler_df = run(sampler_configs, "sampler")
sampler_summary = summarise(sampler_df, BASELINE["name"])
save_table("sampler_summary.csv", sampler_summary)
fmt(sampler_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
order = sampler_summary.sort_values("clip_text", ascending=True)
labels = [SAMPLER_LABELS[s] for s in order.sampler]
y = np.arange(len(order))
axes[0].barh(y, order.clip_text, xerr=order.clip_text_sem, color=PALETTE[0], height=0.6,
             error_kw={"ecolor": INK_2, "capsize": 3})
axes[0].axvline(REAL_REF["clip_text"], color=REF_GREY, linestyle="--", linewidth=1.2)
axes[0].set_yticks(y, labels); axes[0].set_xlabel("CLIP text-image cosine (mean +/- s.e.)")
lo = min(order.clip_text.min(), REAL_REF["clip_text"]) - 0.02
axes[0].set_xlim(lo, max(order.clip_text.max(), REAL_REF["clip_text"]) + 0.01)
axes[0].set_title("Prompt adherence at 30 steps (dashed: real photos)")
axes[1].barh(y, order.delta_clip_text, xerr=order.delta_ci95, color=PALETTE[2], height=0.6,
             error_kw={"ecolor": INK_2, "capsize": 3})
axes[1].axvline(0, color=INK_2, linewidth=1)
axes[1].set_yticks(y, labels); axes[1].set_xlabel("difference from baseline PNDM, paired (95% CI)")
axes[1].set_title("Paired difference from the baseline")
fig.tight_layout(); print("saved ->", save_fig(fig, "sampler_metrics")); plt.show()

fig = image_grid(sampler_configs, [SAMPLER_LABELS[s] for s in SAMPLER_SWEEP],
                 "Samplers at 30 steps, CFG 7.5: same prompt, same starting noise")
print("saved ->", save_fig(fig, "sampler_grid")); plt.show()

The deterministic samplers (DDIM, Euler, LMS, DPM++ 2M, UniPC, PNDM) are all solving the same underlying
equation, so with enough steps they should converge to nearly the same image. **Euler-a** is the odd one
out: it injects fresh noise at every step, so it never converges to one answer and its picture differs in
composition, not just detail.

### 7c. Step count

Three samplers — the baseline PNDM, DDIM (the classic deterministic sampler) and DPM++ 2M (a higher-order
solver designed for few steps) — at 10, 20, 30 and 50 steps. This is the quality-versus-time trade-off.

In [ ]:
STEP_SAMPLERS = ["pndm", "ddim", "dpmpp_2m"]
STEP_COUNTS = [10, 20, 30, 50]
steps_configs = [C(s, n) for s in STEP_SAMPLERS for n in STEP_COUNTS]
steps_df = run(steps_configs, "steps")
steps_summary = summarise(steps_df, BASELINE["name"])
save_table("steps_summary.csv", steps_summary)
fmt(steps_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for i, s in enumerate(STEP_SAMPLERS):
    sub = steps_summary[steps_summary.sampler == s].sort_values("steps")
    axes[0].errorbar(sub.steps, sub.clip_text, yerr=sub.clip_text_sem, color=PALETTE[i], marker="o",
                     capsize=3, label=SAMPLER_LABELS[s])
    axes[1].errorbar(sub.steps, sub.sharpness, yerr=sub.sharpness_sem, color=PALETTE[i], marker="o",
                     capsize=3, label=SAMPLER_LABELS[s])
    axes[2].plot(sub.sec_per_img, sub.clip_text, color=PALETTE[i], marker="o", label=SAMPLER_LABELS[s])
    for _, r in sub.iterrows():
        axes[2].annotate(f"{r.steps}", (r.sec_per_img, r.clip_text), textcoords="offset points",
                         xytext=(4, 4), fontsize=7, color=INK_2)
for ax in axes[:2]:
    ax.set_xticks(sorted({steps_for(n) for n in STEP_COUNTS})); ax.set_xlabel("denoising steps")
axes[0].set_title("Prompt adherence vs number of steps"); axes[0].set_ylabel("CLIP text-image cosine")
axes[0].legend(loc="lower right")
axes[1].axhline(REAL_REF["sharpness"], color=REF_GREY, linestyle="--", linewidth=1.2)
axes[1].set_title("Sharpness vs number of steps (dashed: real photos)")
axes[1].set_ylabel("variance of the Laplacian")
axes[2].set_title("Prompt adherence against wall-clock time (labels: steps)")
axes[2].set_xlabel("seconds per image"); axes[2].set_ylabel("CLIP text-image cosine")
fig.tight_layout(); print("saved ->", save_fig(fig, "steps_metrics")); plt.show()

fig = image_grid([C(s, n) for s in ["pndm", "dpmpp_2m"] for n in STEP_COUNTS],
                 [f"{SAMPLER_LABELS[s]}\n{n} steps" for s in ["pndm", "dpmpp_2m"] for n in STEP_COUNTS],
                 "Step count: PNDM vs DPM++ 2M")
print("saved ->", save_fig(fig, "steps_grid")); plt.show()

### 7d. Schedule shape, timestep spacing and Karras sigmas

Three single-variable experiments, each against its own control:

| Experiment | Held fixed | Varied | Control |
|---|---|---|---|
| **Beta schedule** | DDIM, 30 steps | `scaled_linear` → `linear`, `squaredcos_cap_v2` | `scaled_linear` (what SD 1.5 was trained with) |
| **Timestep spacing** | DDIM, **10** steps — where spacing matters | `leading` → `linspace`, `trailing` | `leading` (SD default) |
| **Karras sigmas** | **20** steps | Euler and DPM++ 2M, with and without Karras spacing | without |

In [ ]:
schedule_experiments = {
    "beta schedule (DDIM, 30 steps)": [
        ("scaled_linear (trained)", C("ddim", 30)),
        ("linear", C("ddim", 30, beta_schedule="linear")),
        ("squaredcos_cap_v2", C("ddim", 30, beta_schedule="squaredcos_cap_v2")),
    ],
    "timestep spacing (DDIM, 10 steps)": [
        ("leading (default)", C("ddim", 10)),
        ("linspace", C("ddim", 10, timestep_spacing="linspace")),
        ("trailing", C("ddim", 10, timestep_spacing="trailing")),
    ],
    "Karras sigmas (20 steps)": [
        ("Euler", C("euler", 20)),
        ("Euler + Karras", C("euler", 20, use_karras_sigmas=True)),
        ("DPM++ 2M", C("dpmpp_2m", 20)),
        ("DPM++ 2M + Karras", C("dpmpp_2m", 20, use_karras_sigmas=True)),
    ],
}
schedule_df = run([c for exp in schedule_experiments.values() for _, c in exp], "schedule")
schedule_summary = summarise(schedule_df, BASELINE["name"])
save_table("schedule_summary.csv", schedule_summary)

# Paired difference of each variant from *its own* control (the first entry of each experiment).
rows = []
all_rows = pd.DataFrame(RESULTS).drop_duplicates(["config", "prompt_idx", "seed"])
for exp, items in schedule_experiments.items():
    ctrl = all_rows[all_rows.config == items[0][1]["name"]].set_index(["prompt_idx", "seed"])
    for label, c in items:
        sub = all_rows[all_rows.config == c["name"]].set_index(["prompt_idx", "seed"])
        d = (sub.clip_text - ctrl.clip_text).dropna()
        rows.append({"experiment": exp, "variant": label, "config": c["name"],
                     "clip_text": sub.clip_text.mean(), "clip_real": sub.clip_real.mean(),
                     "saturation": sub.saturation.mean(), "sec_per_img": sub.sec_per_img.mean(),
                     "delta_vs_control": d.mean(),
                     "delta_ci95": 1.96 * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan})
schedule_vs_control = pd.DataFrame(rows)
save_table("schedule_vs_control.csv", schedule_vs_control)
schedule_vs_control.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (exp, sub) in zip(axes, schedule_vs_control.groupby("experiment", sort=False)):
    y = np.arange(len(sub))[::-1]
    ax.barh(y, sub.delta_vs_control, xerr=sub.delta_ci95, color=PALETTE[0], height=0.55,
            error_kw={"ecolor": INK_2, "capsize": 3})
    ax.axvline(0, color=INK_2, linewidth=1)
    ax.set_yticks(y, sub.variant); ax.set_title(exp)
    ax.set_xlabel("CLIP text score vs the control (paired, 95% CI)")
fig.tight_layout(); print("saved ->", save_fig(fig, "schedule_metrics")); plt.show()

for exp, items in schedule_experiments.items():
    fig = image_grid([c for _, c in items], [l for l, _ in items], exp)
    print("saved ->", save_fig(fig, "schedule_grid_" + exp.split(" (")[0].replace(" ", "_"))); plt.show()

### 7e. DDPM: 1,000 steps vs 20

DDPM is the original diffusion sampler — it walks every one of the 1,000 training timesteps, adding fresh
noise each time. It is too slow for the sweep, so it runs once, on two prompts, next to the fast samplers.
The point is the cost column: modern solvers reach comparable images in 2% of the steps.

In [ ]:
DDPM_PROMPTS = list(range(min(2, N_PROMPTS)))
# SD 1.5's scheduler config has steps_offset = 1 with "leading" spacing, which shifts every timestep up
# by one. At 1,000 steps that produces t = 1000, one past the end of the 1,000-entry schedule (t runs
# 0-999). "trailing" spacing visits exactly 999 ... 0, every training timestep once.
ddpm_configs = [("DDPM, 1000 steps", C("ddpm", 1000, timestep_spacing="trailing")), ("DDIM, 50 steps", C("ddim", 50)),
                ("DPM++ 2M, 20 steps", C("dpmpp_2m", 20))]
ddpm_df = run([c for _, c in ddpm_configs], "ddpm_demo", prompt_ids=DDPM_PROMPTS, seeds=[SEEDS[0]])
ddpm_summary = summarise(ddpm_df)
ddpm_summary.insert(0, "label", [l for l, _ in ddpm_configs])
save_table("ddpm_demo.csv", ddpm_summary)
display(ddpm_summary[["label", "steps", "clip_text", "clip_real", "sec_per_img"]].round(3))

fig = image_grid([c for _, c in ddpm_configs],
                 [f"{l}\n{t:.1f} s/img" for (l, _), t in zip(ddpm_configs, ddpm_summary.sec_per_img)],
                 "DDPM (1,000 steps) vs fast solvers", prompt_ids=DDPM_PROMPTS, cell=2.6)
print("saved ->", save_fig(fig, "ddpm_demo")); plt.show()

---
## 8. Results table and the selected configuration

Every configuration from sections 5–7, on the same 24 (prompt, seed) pairs. The DDPM demonstration is
left out: it used only two prompts.

**Selection rule**, written down before looking at the numbers so it is not cherry-picked:

1. Take the highest mean `clip_text`.
2. Every configuration within one standard error of it counts as tied — differences that small are noise
   at 24 images.
3. Drop the tied ones that look worse than the baseline in ways CLIP does not score: **over-saturated**
   (saturation more than 0.10 above the baseline's) or **blurrier** (sharpness below 90% of the
   baseline's).
4. Among what is left, prefer the fewest steps (cheapest), then the lowest guidance.

Step 3 was added after the first full run. Without it, the rule picked DDIM at 10 steps: tied on CLIP
score and nearly 3× faster, but visibly softer than the baseline. CLIP checks *what* is in the picture,
not how crisply it is drawn.

`FINAL_OVERRIDE` lets you pick a configuration by name instead.

In [ ]:
FINAL_OVERRIDE = None     # e.g. "dpmpp_2m_s20_g7.5" to force a choice

sweep_df = (pd.DataFrame(RESULTS).query("group not in ['ddpm_demo', 'final']")
            .drop_duplicates(["config", "prompt_idx", "seed"]))
all_summary = summarise(sweep_df.assign(group="sweep"), BASELINE["name"]).drop(columns="group")
all_summary = all_summary.sort_values("clip_text", ascending=False).reset_index(drop=True)
save_table("all_configs_summary.csv", all_summary)

top = all_summary.iloc[0]
base_row = all_summary.set_index("config").loc[BASELINE["name"]]
tied = all_summary[all_summary.clip_text >= top.clip_text - top.clip_text_sem].copy()
tied["oversaturated"] = tied.saturation > base_row.saturation + 0.10
tied["blurrier"] = tied.sharpness < 0.9 * base_row.sharpness
eligible = tied[~tied.oversaturated & ~tied.blurrier]
if eligible.empty:                          # nothing tied beats the checks: keep the baseline
    eligible = all_summary[all_summary.config == BASELINE["name"]]
ranked = eligible.sort_values(["steps", "cfg"])
BEST_NAME = FINAL_OVERRIDE or ranked.iloc[0].config
display(tied[["config", "clip_text", "saturation", "sharpness", "steps", "oversaturated", "blurrier"]].round(4))
best_row = all_summary.set_index("config").loc[BEST_NAME]
BEST = next(c for c in [BASELINE] + cfg_configs + sampler_configs + steps_configs +
            [c for exp in schedule_experiments.values() for _, c in exp] if c["name"] == BEST_NAME)

print(f"{len(all_summary)} configurations. Top clip_text: {top.config} ({top.clip_text:.4f} +/- {top.clip_text_sem:.4f})")
print(f"{len(tied)} within one standard error of it: {', '.join(tied.config)}")
print(f"\nSELECTED: {BEST_NAME}" + ("  (manual override)" if FINAL_OVERRIDE else ""))
print(f"  clip_text  {best_row.clip_text:.4f}   vs baseline: {best_row.delta_clip_text:+.4f} "
      f"(95% CI +/- {best_row.delta_ci95:.4f})")
print(f"  sharpness  {best_row.sharpness:.0f}   (baseline {base_row.sharpness:.0f}, real photos {REAL_REF['sharpness']:.0f})")
print(f"  saturation {best_row.saturation:.3f}   time {best_row.sec_per_img:.2f} s/image "
      f"(baseline {all_summary.set_index('config').loc[BASELINE['name']].sec_per_img:.2f})")
save_table("selected_config.json", {**BEST, "selection": "override" if FINAL_OVERRIDE else "rule",
                                    "metrics": best_row.to_dict()})
fmt(all_summary)

In [ ]:
# Where every configuration sits: adherence against cost, coloured by experiment family.
fam = {"cfg": "CFG sweep", "sampler": "samplers @30", "steps": "step counts", "schedule": "schedule shape/spacing",
       "baseline": "baseline"}
first_group = sweep_df.drop_duplicates("config").set_index("config").group
fig, ax = plt.subplots(figsize=(9, 4.8))
for i, (g, label) in enumerate(fam.items()):
    sub = all_summary[all_summary.config.map(first_group) == g]
    if len(sub):
        ax.scatter(sub.sec_per_img, sub.clip_text, s=42, color=PALETTE[i], label=label,
                   edgecolors="#fcfcfb", linewidths=1.5, zorder=3)
for name, tag in [(BASELINE["name"], "baseline"), (BEST_NAME, "selected")]:
    r = all_summary.set_index("config").loc[name]
    ax.annotate(f"{tag}: {name}", (r.sec_per_img, r.clip_text), textcoords="offset points", xytext=(6, -12),
                fontsize=8, color=INK, arrowprops={"arrowstyle": "-", "color": INK_2, "lw": 0.8})
ax.axhline(REAL_REF["clip_text"], color=REF_GREY, linestyle="--", linewidth=1.2)
ax.set_xlabel("seconds per image"); ax.set_ylabel("CLIP text-image cosine")
ax.set_title("Every configuration: prompt adherence vs cost (dashed: real photos)")
ax.legend(loc="lower right")
fig.tight_layout(); print("saved ->", save_fig(fig, "all_configs_tradeoff")); plt.show()

---
## 9. First set of generated images

### 9a. Final set

The selected configuration on all 12 prompts × 4 starting noises (48 images), with the baseline run on
the same 48 for a like-for-like comparison.

In [ ]:
final_configs = [BASELINE] if BEST_NAME == BASELINE["name"] else [BASELINE, BEST]
final_df = run(final_configs, "final", seeds=FINAL_SEEDS)
final_summary = summarise(final_df, BASELINE["name"])
save_table("final_summary.csv", final_summary)
display(fmt(final_summary))

fig, axes = plt.subplots(len(FINAL_SEEDS) + 1, N_PROMPTS,
                         figsize=(1.7 * N_PROMPTS, 1.8 * (len(FINAL_SEEDS) + 1) + 0.5), squeeze=False)
for p in range(N_PROMPTS):
    imgs = [Image.open(prompts_df.loc[p, "real_path"])] + [Image.open(image_path(BEST, p, s)) for s in FINAL_SEEDS]
    for r, img in enumerate(imgs):
        ax = axes[r, p]; ax.imshow(img); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for sp in ax.spines.values():
            sp.set_visible(False)
    axes[0, p].set_title(f"[{p}]", fontsize=8)
for r, lab in enumerate(["real"] + [f"seed {s}" for s in FINAL_SEEDS]):
    axes[r, 0].set_ylabel(lab, fontsize=8, color=INK_2)
fig.suptitle(f"First recorded set -- {BEST_NAME}: real photo on top, four generations below",
             fontsize=11, color=INK)
fig.tight_layout(); print("saved ->", save_fig(fig, "final_set_grid")); plt.show()

### 9b. Baseline vs selected, side by side

In [ ]:
ids = list(range(min(6, N_PROMPTS)))
fig, axes = plt.subplots(len(ids), 3, figsize=(7.5, 2.5 * len(ids)), squeeze=False)
for r, p in enumerate(ids):
    imgs = [Image.open(prompts_df.loc[p, "real_path"]), Image.open(image_path(BASELINE, p, SEEDS[0])),
            Image.open(image_path(BEST, p, SEEDS[0]))]
    for c, (img, head) in enumerate(zip(imgs, ["real photo", "baseline", "selected"])):
        ax = axes[r, c]; ax.imshow(img); ax.axis("off")
        if r == 0:
            ax.set_title(head, fontsize=9)
    axes[r, 1].text(0, -0.04, PROMPTS[p][:70] + ("..." if len(PROMPTS[p]) > 70 else ""),
                    transform=axes[r, 1].transAxes, fontsize=6.5, color=INK_2, va="top")
fig.suptitle(f"Baseline ({BASELINE['name']}) vs selected ({BEST_NAME})", fontsize=10)
fig.tight_layout(); print("saved ->", save_fig(fig, "baseline_vs_selected")); plt.show()

### 9c. Inside one generation: the denoising trajectory

What the latent looks like as the sampler walks from pure noise to an image — DDIM, 30 steps, one prompt.
Each frame decodes the current latent through the VAE.

In [ ]:
traj_steps = steps_for(30)
keep = sorted(set(np.linspace(0, traj_steps - 1, min(8, traj_steps)).round().astype(int)))
frames = {}

def capture(pipeline, i, t, kwargs):
    if i in keep:
        with torch.no_grad():
            lat = kwargs["latents"][:1] / pipeline.vae.config.scaling_factor
            img = pipeline.vae.decode(lat.to(pipeline.vae.dtype)).sample
        frames[i] = (int(t), ((img[0].float().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy())
    return kwargs

pipe.scheduler = make_scheduler("ddim")
p = 0
_ = pipe(prompt_embeds=PROMPT_EMB[[p]], negative_prompt_embeds=NEG_EMB, latents=start_latent(p, SEEDS[0]).to(GEN_DEVICE, GEN_DTYPE),
         num_inference_steps=traj_steps, guidance_scale=7.5, height=RESOLUTION, width=RESOLUTION,
         callback_on_step_end=capture, callback_on_step_end_tensor_inputs=["latents"])

fig, axes = plt.subplots(1, len(frames), figsize=(2.1 * len(frames), 2.6))
for ax, (i, (t, img)) in zip(np.atleast_1d(axes), sorted(frames.items())):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"step {i + 1}\n(t = {t})", fontsize=8)
fig.suptitle(f"Denoising trajectory, DDIM {traj_steps} steps: \"{PROMPTS[p][:80]}\"", fontsize=10)
fig.tight_layout(); print("saved ->", save_fig(fig, "denoising_trajectory")); plt.show()

---
## 10. Export

Everything is under `m2_outputs/`, and zipped into `m2_outputs.zip` next to it for one-click download
from the Kaggle *Output* tab.

| Folder | Contents |
|---|---|
| `generations/<config>/` | every image, named `p<prompt>_s<seed>.png`, plus `timing.json` |
| `plots/` | every figure shown above |
| `tables/` | `prompts.csv`, `integration_check.json`, per-experiment summaries, `all_configs_summary.csv`, `results_per_image.csv`, `selected_config.json`, `run_info.json` |

In [ ]:
run_info = {
    "quick_run": QUICK_RUN, "seed": SEED, "n_prompts": N_PROMPTS, "seeds": SEEDS, "final_seeds": FINAL_SEEDS,
    "resolution": RESOLUTION, "sd_repo": SD_REPO, "text_encoder": TEXT_ENCODER_REPO, "scorer": SCORER_REPO,
    "baseline": BASELINE["name"], "selected": BEST_NAME, "n_configs": int(len(all_summary)),
    "n_images_scored": int(len(pd.DataFrame(RESULTS).drop_duplicates(["config", "prompt_idx", "seed"]))),
    "real_photo_reference": REAL_REF,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "diffusers": diffusers.__version__, "transformers": transformers.__version__},
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
save_table("run_info.json", run_info)

zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
n_png = sum(1 for _ in GEN_DIR.rglob("*.png"))
print(f"{n_png} generated images across {len(list(GEN_DIR.iterdir()))} configurations")
print(f"zip: {zip_path}  ({Path(zip_path).stat().st_size / 1024**2:.0f} MB)")

---
## Summary

The tables and plots above are the findings; this is how to read them for the write-up.

- **Integration** — section 3 / `integration_check.json`: our separately loaded CLIP encoder produces the
  same conditioning tensors as the pipeline's, and `prompt_embeds` reproduces `prompt=`. The pipeline
  never sees a string after that point.
- **Baseline** — `baseline_summary.csv`: PNDM, 30 steps, CFG 7.5 on 24 fixed (prompt, noise) pairs, with
  the real photos' own score as the ceiling.
- **CFG** — `cfg_sweep_metrics.png`: prompt adherence rises from *w* = 1 and flattens; saturation keeps
  climbing past the real photos; guidance rescale pulls it back. Time per image is flat for every *w* > 1.
- **Noise schedule** — `sampler_metrics.png`, `steps_metrics.png`, `schedule_metrics.png`: which solvers
  hold up at low step counts, what swapping the trained beta schedule costs, and where `trailing` spacing
  and Karras sigmas help.
- **First image set** — `final_set_grid.png`, `baseline_vs_selected.png`, `denoising_trajectory.png`, with
  the selected configuration and its paired difference from the baseline in `final_summary.csv`.